In [2]:
rename_dict = {
    0: 'ip_num_low',
    1: 'ip_num_high',
    2: 'proxytype',
    3: 'country_code',
    4: 'country_name',
    5: 'region',
    6: 'city',
    7: 'isp',
    8: 'domain',
    9: 'usage_type',
    10: 'asn',
    11: 'company',
    12: 'threat_level',
    13: 'threat_type',
    14: 'provider'
}

region_name = "us-east-1"

In [32]:
import boto3
import os
import requests
import pandas as pd
import io
import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, udf, current_timestamp
from pyspark.sql.types import StringType

def extract_and_save_to_local():
    """
    Downloads data from a URL and saves it to a local file.
    
    Returns:
        str: The path to the saved file.
    """
    # Replace with your actual token and database code
    token = os.getenv('IP2LOCATION_TOKEN')
    database_code = 'PX11LITECSV'
    dataname = 'px11'

    # Construct the URL
    url = f"https://www.ip2location.com/download/?token={token}&file={database_code}"

    # output_path imported as constant
    # Set the destination path for the downloaded file
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    try:
        # Send GET request to download the file
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an error for bad responses

        # Write the file in chunks to handle large downloads
        with open(output_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    file.write(chunk)

        print(f"Download successful! File saved to {output_path}")
        return output_path

    except requests.exceptions.RequestException as e:
        print(f"Download failed: {e}")
        return None


def execute_sql_from_file_with_spark(sql_file_path, catalog_name, region_name):
    """
    Executes a SQL file to create an Iceberg table using Apache Spark.

    Args:
        sql_file_path (str): Path to the SQL file containing the table creation statement.
        catalog_name (str): The Iceberg catalog name in Spark.
        warehouse_location (str): The S3 warehouse location for Iceberg tables.
    """
    # Initialize Spark session with Iceberg configurations
    spark = SparkSession.builder \
        .appName("IcebergTableCreation") \
        .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{catalog_name}.type", "glue") \
        .config(f"spark.sql.catalog.{catalog_name}.warehouse", f"s3://{catalog_name}/") \
        .config(f"spark.sql.catalog.{catalog_name}.region", region_name) \
        .getOrCreate()

    # Read SQL from file
    with open(sql_file_path, 'r') as f:
        sql = f.read()

    # Execute SQL
    spark.sql(sql)
    print("Table creation executed successfully.")

def transform(spark_df, rename_dict):
    """
    Transforms a Spark DataFrame:
    - Converts IP numbers to addresses.
    - Renames columns to lowercase per rename_dict.
    - Creates low and high IP address columns.
    - Drops the 'proxytype' column.
    
    Args:
        spark_df (pyspark.sql.DataFrame): The input Spark DataFrame.
        rename_dict (dict): A dictionary mapping old column names to new column names.
    
    Returns:
        pyspark.sql.DataFrame: The transformed Spark DataFrame.
    """
    # Define a UDF to convert IP numbers to addresses
    @udf(StringType())
    def ip_number_to_address(ip_number):
        return f"{(ip_number >> 24) & 255}.{(ip_number >> 16) & 255}.{(ip_number >> 8) & 255}.{ip_number & 255}" if ip_number is not None else None

    # Rename columns using the rename_dict
    for old_col, new_col in rename_dict.items():
        spark_df = spark_df.withColumnRenamed(old_col, new_col)

    # Add 'ip_low' and 'ip_high' columns by applying the UDF
    spark_df = spark_df.withColumn('ip_low', ip_number_to_address(col('ip_num_low')))
    spark_df = spark_df.withColumn('ip_high', ip_number_to_address(col('ip_num_high')))
    spark_df = spark_df.withColumn('updated_at', current_timestamp())

    # Drop the 'proxytype' column
    if 'proxytype' in spark_df.columns:
        spark_df = spark_df.drop('proxytype')

    return spark_df


def save_to_iceberg_table(df, table_name, catalog_name, bucket_name, object_key, aws_access_key_id=None, aws_secret_access_key=None, region_name=None):
    """
    Saves a Pandas DataFrame as a Parquet file to an S3 bucket.

    Args:
        df (pd.DataFrame): The DataFrame to save.
        table_name (str): The Iceberg table name in AWS Glue.
        catalog_name (str): The AWS Glue catalog name for Iceberg.
        bucket_name (str): The name of the S3 bucket.
        object_key (str): The S3 object key (path within the bucket).
        aws_access_key_id (str): AWS access key ID (optional, uses environment variables if not provided).
        aws_secret_access_key (str): AWS secret access key (optional, uses environment variables if not provided).
        region_name (str): AWS region name (optional, uses default if not provided).
    """
    try:
        # Step 1: Initialize Spark Session with Iceberg Support
        spark = SparkSession.builder \
            .appName("IcebergWriter") \
            .config("spark.sql.catalog." + catalog_name, "org.apache.iceberg.spark.SparkCatalog") \
            .config("spark.sql.catalog." + catalog_name + ".type", "hive") \
            .config("spark.sql.catalog." + catalog_name + ".warehouse", f"s3://{bucket_name}/") \
            .config("spark.hadoop.fs.s3a.access.key", aws_access_key_id) \
            .config("spark.hadoop.fs.s3a.secret.key", aws_secret_access_key) \
            .config("spark.sql.catalog." + catalog_name + ".uri", f"glue://{region_name}") \
            .getOrCreate()

        # Step 2: Convert Pandas DataFrame to Spark DataFrame
        spark_df = spark.createDataFrame(df)

        # Step 3: Write Data to Iceberg Table
        spark_df.writeTo(f"{catalog_name}.{table_name}") \
            .append()  # Use "overwrite" if you want to replace data instead

    except Exception as e:
        print(f"Failed to save to Iceberg table: {e}")

    finally:
        # Stop the Spark session
        if 'spark' in locals():
            spark.stop()

In [21]:
spark_df = extract_to_spark_dataframe()

Constructed url successfully


ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 2


In [25]:
# Grab token from K8s Secrets.
token = os.getenv("TOKEN", "No Token Found")
database_code = 'PX11LITECSV'
dataname = 'px11'
# Construct the URL for data pull
url = f"https://www.ip2location.com/download/?token={token}&file={database_code}"
print("Constructed url successfully")

try:
    # Send GET request to download the file
    response = requests.get(url, stream=True)
    response.raise_for_status()  # Raise an error for bad responses
    print(response.content[:500].decode('utf-8', errors='replace'))
    data = io.BytesIO(response.content)  # Read the content into memory
except:
    pass
    # Read the response content into a Pandas DataFrame (assuming CSV format)
    
    
    
#     pandas_df = pd.read_csv(data, encoding='latin-1')  # Adjust to appropriate reader (CSV, JSON, etc.)

#     print("Download and conversion to Pandas DataFrame successful!")

#     # Convert Pandas DataFrame to Spark DataFrame
#     spark = SparkSession.builder.appName("ExtractToSparkDF").getOrCreate()
#     spark_df = spark.createDataFrame(pandas_df)

#     print("Conversion to Spark DataFrame successful!")

# except requests.exceptions.RequestException as e:
#     print(f"Download failed: {e}")

Constructed url successfully
THIS FILE CAN ONLY BE DOWNLOADED 5 TIMES WITHIN 24 HOURS


In [27]:
data.seek(0)  # Move to the start of the BytesIO object
print(data.read(500).decode('utf-8', errors='replace'))

THIS FILE CAN ONLY BE DOWNLOADED 5 TIMES WITHIN 24 HOURS


In [28]:
print(response.text)

THIS FILE CAN ONLY BE DOWNLOADED 5 TIMES WITHIN 24 HOURS


In [41]:
print(spark.conf.get("spark.jars", "No JARs configured"))

No JARs configured


In [34]:
sql_file_path="/home/eandrews/projects/de-proj-1/docker/bronze_etl/sql/create_catalog.sql"
catalog_name="bronze_cat"
region_name="us-east-1"

In [35]:
spark = SparkSession.builder \
        .appName("IcebergTableCreation") \
        .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{catalog_name}.type", "glue") \
        .config(f"spark.sql.catalog.{catalog_name}.warehouse", f"s3://{catalog_name}/") \
        .config(f"spark.sql.catalog.{catalog_name}.region", region_name) \
        .getOrCreate()

25/01/28 15:46:59 WARN Utils: Your hostname, ubuntu1 resolves to a loopback address: 127.0.1.1; using 192.168.0.46 instead (on interface wlp2s0)


In [36]:
spark

In [42]:
spark.stop()

In [40]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS bronze_cat.ip_extract")

In [39]:
with open(sql_file_path, 'r') as f:
    sql = f.read()

# Execute SQL
spark.sql(sql)
print("Table creation executed successfully.")

AnalysisException: [REQUIRES_SINGLE_PART_NAMESPACE] spark_catalog requires a single-part namespace, but got `glue_catalog_name`.`database_name`.

In [ ]:
execute_sql_from_file_with_spark(sql_file_path="/home/eandrews/projects/de-proj-1/docker/bronze_etl/sql/create_catalog.sql",
                                 catalog_name="bronze_cat",
                                 region_name="us-east-1")